In [48]:
# path for pijax

import sys
sys.path.append("/mnt/lustre-grete/usr/u12045/vla/duci/openpi/src")
sys.path.append("/mnt/lustre-grete/usr/u12045/vla/duci/openpi")
import os
os.environ['HF_HOME'] = "/mnt/lustre-grete/usr/u12045/vla/hf_cache"
from dotenv import load_dotenv
load_dotenv()

True

In [49]:
# path for pitorch

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["PATH"] = "/mnt/lustre-grete/usr/u12045/projects/LLAVA-Med/envs/lerobot/bin:" + os.environ.get("PATH", "")
os.environ["HF_HOME"] = "/mnt/lustre-grete/usr/u12045/vla/hf_cache"
os.environ["TMPDIR"] = "/mnt/lustre-grete/usr/u12045/vla/cache"
os.environ["PYTHONPATH"] = "/mnt/lustre-grete/usr/u12045/vla/duci/VLA-Humanoid:" + os.environ.get("PYTHONPATH", "")
import sys
sys.path.insert(0, "/mnt/lustre-grete/usr/u12045/vla/duci/VLA-Humanoid")

os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.5"

# Compare dataloader

## Pijax load dataset

In [50]:
import dataclasses

import jax

from openpi.models import model as _model
from openpi.policies import droid_policy
from openpi.policies import policy_config as _policy_config
from openpi.shared import download
from openpi.training import config as _config
from openpi.training import data_loader as _data_loader
from pathlib import Path


%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [51]:
import openpi
print(openpi)

<module 'openpi' from '/mnt/lustre-grete/usr/u12045/vla/duci/openpi/src/openpi/__init__.py'>


In [52]:
from collections.abc import Iterator, Sequence
import multiprocessing
import os
import typing
from typing import Protocol, SupportsIndex, TypeVar

import jax
import jax.numpy as jnp
import lerobot.common.datasets.lerobot_dataset as lerobot_dataset
import numpy as np
import torch

import openpi.models.model as _model
import openpi.training.config as _config
import openpi.transforms as _transforms

T_co = TypeVar("T_co", covariant=True)

class Dataset(Protocol[T_co]):
    """Interface for a dataset with random access."""

    def __getitem__(self, index: SupportsIndex) -> T_co:
        raise NotImplementedError("Subclasses of Dataset should implement __getitem__.")

    def __len__(self) -> int:
        raise NotImplementedError("Subclasses of Dataset should implement __len__.")

class TransformedDataset(Dataset[T_co]):
    def __init__(self, dataset: Dataset, transforms: Sequence[_transforms.DataTransformFn]):
        self._dataset = dataset
        self._transform = _transforms.compose(transforms)

    def __getitem__(self, index: SupportsIndex) -> T_co:
        return self._transform(self._dataset[index])

    def __len__(self) -> int:
        return len(self._dataset)


class FakeDataset(Dataset):
    def __init__(self, model_config: _model.BaseModelConfig, num_samples: int):
        self._num_samples = num_samples
        self._observation_spec, self._action_spec = model_config.inputs_spec()

    def __getitem__(self, index: SupportsIndex) -> dict:
        rng = jax.random.key(index.__index__())

        def make_from_spec(spec: jax.ShapeDtypeStruct):
            nonlocal rng
            rng, data_rng = jax.random.split(rng)
            # Remove the batch dimension.
            shape = spec.shape[1:]
            if spec.dtype == jnp.float32:
                return jax.random.uniform(data_rng, shape=shape, minval=-1.0, maxval=1.0)
            if spec.dtype == jnp.int32:
                return jax.random.randint(data_rng, shape=shape, minval=0, maxval=2048)
            return jnp.zeros(shape=shape, dtype=spec.dtype)

        observation = jax.tree.map(make_from_spec, self._observation_spec)
        action = jax.tree.map(make_from_spec, self._action_spec)

        return {
            **observation.to_dict(),
            "actions": action,
        }

    def __len__(self) -> int:
        return self._num_samples

In [53]:
def create_dataset(data_config: _config.DataConfig, model_config: _model.BaseModelConfig) -> Dataset:
    """Create a dataset for training."""
    
    repo_id = data_config.repo_id
    if repo_id is None:
        raise ValueError("Repo ID is not set. Cannot create dataset.")
    if repo_id == "fake":
        return FakeDataset(model_config, num_samples=1024)

    dataset_meta = lerobot_dataset.LeRobotDatasetMetadata(repo_id)
    dataset = lerobot_dataset.LeRobotDataset(
        data_config.repo_id,
        delta_timestamps={
            key: [t / dataset_meta.fps for t in range(model_config.action_horizon)]
            for key in data_config.action_sequence_keys
        },
    )

    if data_config.prompt_from_task:
        dataset = TransformedDataset(dataset, [_transforms.PromptFromLeRobotTask(dataset_meta.tasks)])

    return dataset

In [54]:
config = _config.get_config("pi0_calvin_50%_joint")

from pathlib import Path
assets_dirs = Path('/mnt/lustre-grete/usr/u12045/vla/duci/openpi/assets/pi0_calvin_50%_joint')
data_config = config.data.create(assets_dirs, config.model)


In [55]:
# jax_stats = {}
# jax_stats['state'] = {}
# jax_stats['state']['mean'] = data_config.norm_stats['state'].mean
# jax_stats['state']['std'] = data_config.norm_stats['state'].std
# jax_stats['joint_actions'] = {}
# jax_stats['joint_actions']['mean'] = data_config.norm_stats['actions'].mean
# jax_stats['joint_actions']['std'] = data_config.norm_stats['actions'].std
# np.save("norm_stats_jax.npy", jax_stats)

In [56]:
jaxdataset = create_dataset(data_config, config.model)


The dataset you requested (ducido/calvin_task_D_D_scale_50_lerobo_format) is in 2.0 format.
While current version of LeRobot is backward-compatible with it, the version of your dataset still uses global
stats instead of per-episode stats. Update your dataset stats to the new format using this command:
```
python lerobot/common/datasets/v21/convert_dataset_v20_to_v21.py --repo-id=ducido/calvin_task_D_D_scale_50_lerobo_format
```

If you encounter a problem, contact LeRobot maintainers on [Discord](https://discord.com/invite/s3KuuzsPFb)
or open an [issue on GitHub](https://github.com/huggingface/lerobot/issues/new/choose).

The dataset you requested (ducido/calvin_task_D_D_scale_50_lerobo_format) is in 2.0 format.
While current version of LeRobot is backward-compatible with it, the version of your dataset still uses global
stats instead of per-episode stats. Update your dataset stats to the new format using this command:
```
python lerobot/common/datasets/v21/convert_dataset_v20_to_v21.p

## Pitorch load dataset

In [57]:

import logging
import time
from contextlib import nullcontext
from pprint import pformat
from typing import Any

import torch
from termcolor import colored
from torch.amp import GradScaler
from torch.optim import Optimizer

from lerobot.common.datasets.factory import make_dataset
from lerobot.common.datasets.sampler import EpisodeAwareSampler
from lerobot.common.datasets.utils import cycle
from lerobot.common.envs.factory import make_env
from lerobot.common.optim.factory import make_optimizer_and_scheduler
from lerobot.common.policies.factory import make_policy
from lerobot.common.policies.pretrained import PreTrainedPolicy
from lerobot.common.policies.utils import get_device_from_parameters
from lerobot.common.utils.logging_utils import AverageMeter, MetricsTracker
from lerobot.common.utils.random_utils import set_seed
from lerobot.common.utils.train_utils import (
    get_step_checkpoint_dir,
    get_step_identifier,
    load_training_state,
    save_checkpoint,
    update_last_checkpoint,
)
from lerobot.common.utils.utils import (
    format_big_number,
    get_safe_torch_device,
    has_method,
    init_logging,
)
from lerobot.common.utils.wandb_utils import WandBLogger
from lerobot.configs import parser
from lerobot.configs.train import TrainPipelineConfig
from lerobot.scripts.eval import eval_policy
from lerobot.common.constants import ACTION, OBS_STATE

import json
from dotenv import load_dotenv
load_dotenv()


%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [58]:
from lerobot.configs.policies import PreTrainedConfig
from lerobot.configs.default import DatasetConfig, EvalConfig, WandBConfig
from lerobot.common.datasets.transforms import ImageTransformsConfig

with open("../configs/calvin_config/default.json") as f:
    data = json.load(f)
# /mnt/lustre-grete/usr/u12045/vla/hf_cache/lerobot/binhng/libero_spatial_mask_debug_noops_lerobot
cfg = TrainPipelineConfig(
    policy=PreTrainedConfig.from_pretrained("lerobot/pi0"),
    dataset=DatasetConfig(
        repo_id="ducido/calvin_task_D_D_scale_50_lerobo_format",
        image_transforms=ImageTransformsConfig.from_dict(data["dataset"]['image_transforms']),
    ),
    wandb=WandBConfig(
        project="lerobot",
        entity="lerobot",
    ),
)
cfg.validate()
# logging.info(pformat(cfg.to_dict()))

In [59]:
cfg.policy.chunk_size = 50
cfg.policy.n_action_steps = 50
cfg.batch_size = 1
cfg.wandb.enable = False
cfg.dataset.image_transforms.enable = False
cfg.log_freq = 20

In [60]:
if cfg.wandb.enable and cfg.wandb.project:
    wandb_logger = WandBLogger(cfg)
else:
    wandb_logger = None
    logging.info(colored("Logs will be saved locally.", "yellow", attrs=["bold"]))

if cfg.seed is not None:
    set_seed(cfg.seed)

device = get_safe_torch_device(cfg.policy.device, log=True)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

logging.info("Creating dataset")
torchdataset = make_dataset(cfg)

The dataset you requested (ducido/calvin_task_D_D_scale_50_lerobo_format) is in 2.0 format.
While current version of LeRobot is backward-compatible with it, the version of your dataset still uses global
stats instead of per-episode stats. Update your dataset stats to the new format using this command:
```
python lerobot/common/datasets/v21/convert_dataset_v20_to_v21.py --repo-id=ducido/calvin_task_D_D_scale_50_lerobo_format
```

If you encounter a problem, contact LeRobot maintainers on [Discord](https://discord.com/invite/s3KuuzsPFb)
or open an [issue on GitHub](https://github.com/huggingface/lerobot/issues/new/choose).

The dataset you requested (ducido/calvin_task_D_D_scale_50_lerobo_format) is in 2.0 format.
While current version of LeRobot is backward-compatible with it, the version of your dataset still uses global
stats instead of per-episode stats. Update your dataset stats to the new format using this command:
```
python lerobot/common/datasets/v21/convert_dataset_v20_to_v21.p

In [61]:
i = 0
jaxsample = jaxdataset[i]
torchsample = torchdataset[i]

In [62]:
all_keys = torchsample.keys()

In [63]:
def compare_key(key, d1, d2):
    if isinstance(d2[key], str):
        return d1[key] == d2[key]
    return torch.all(d1[key] == d2[key])

In [64]:
for key in torchsample:
    try:
        print(key, compare_key(key, jaxsample, torchsample))
    except:
        print(key)

image tensor(True)
wrist_image tensor(True)
state tensor(True)
actions tensor(True)
rel_actions tensor(True)
joint_actions tensor(True)
rel_joint_actions tensor(True)
depth_gripper tensor(True)
depth_static tensor(True)
scene_obs tensor(True)
timestamp tensor(True)
frame_index tensor(True)
episode_index tensor(True)
index tensor(True)
task_index tensor(True)
joint_actions_is_pad tensor(True)
task True


## When norm

## jax

In [65]:
class TransformedDataset(Dataset[T_co]):
    def __init__(self, dataset: Dataset, transforms: Sequence[_transforms.DataTransformFn]):
        self._dataset = dataset
        self._transform = _transforms.compose(transforms)

    def __getitem__(self, index: SupportsIndex) -> T_co:
        return self._transform(self._dataset[index])

    def __len__(self) -> int:
        return len(self._dataset)

def transform_dataset(dataset: Dataset, data_config: _config.DataConfig, *, skip_norm_stats: bool = False) -> Dataset:
    """Transform the dataset by applying the data transforms."""
    norm_stats = {}
    if data_config.repo_id != "fake" and not skip_norm_stats:
        if data_config.norm_stats is None:
            raise ValueError(
                "Normalization stats not found. "
                "Make sure to run `scripts/compute_norm_stats.py --config-name=<your-config>`."
            )
        norm_stats = data_config.norm_stats
    print([*data_config.model_transforms.inputs])
    return TransformedDataset(
        dataset,
        [
            *data_config.repack_transforms.inputs,
            *data_config.data_transforms.inputs,
            _transforms.Normalize(norm_stats, use_quantiles=data_config.use_quantile_norm),
            *data_config.model_transforms.inputs,
        ],
    )
skip_norm_stats = False
jax_dataset_trans = transform_dataset(jaxdataset, data_config, skip_norm_stats=skip_norm_stats)

[InjectDefaultPrompt(prompt=None), ResizeImages(height=224, width=224), TokenizePrompt(tokenizer=<openpi.models.tokenizer.PaligemmaTokenizer object at 0x1486fb817ed0>)]


In [134]:
class TorchDataLoader:
    def __init__(
        self,
        dataset,
        local_batch_size: int,
        *,
        sharding: jax.sharding.Sharding | None = None,
        shuffle: bool = False,
        num_batches: int | None = None,
        num_workers: int = 0,
        seed: int = 0,
    ):
        """Create a PyTorch data loader.

        Args:
            dataset: The dataset to load.
            local_batch_size: The local batch size for each process.
            sharding: The sharding to use for the data loader.
            shuffle: Whether to shuffle the data.
            num_batches: If provided, determines the number of returned batches. If the
                number is larger than the number of batches in the dataset, the data loader
                will loop over the dataset. If not provided, will iterate over the dataset
                indefinitely.
            num_workers: The number of worker processes to use. If zero, the data loader will
                execute in the main process.
            seed: The seed to use for shuffling the data.
        """
        if jax.process_count() > 1:
            raise NotImplementedError("Data loading with multiple processes is not supported.")

        if len(dataset) < local_batch_size:
            raise ValueError(f"Local batch size ({local_batch_size}) is larger than the dataset size ({len(dataset)}).")

        if sharding is None:
            # Use data parallel sharding by default.
            sharding = jax.sharding.NamedSharding(
                jax.sharding.Mesh(jax.devices(), ("B",)),
                jax.sharding.PartitionSpec("B"),
            )

        self._sharding = sharding
        self._num_batches = num_batches

        mp_context = None
        if num_workers > 0:
            mp_context = multiprocessing.get_context("spawn")

        generator = torch.Generator()
        generator.manual_seed(seed)
        self._data_loader = torch.utils.data.DataLoader(
            typing.cast(torch.utils.data.Dataset, dataset),
            batch_size=local_batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            multiprocessing_context=mp_context,
            persistent_workers=num_workers > 0,
            collate_fn=_collate_fn,
            worker_init_fn=_worker_init_fn,
            drop_last=True,
            generator=generator,
        )

    @property
    def torch_loader(self) -> torch.utils.data.DataLoader:
        return self._data_loader

    def __iter__(self):
        num_items = 0
        while True:
            data_iter = iter(self._data_loader)
            while True:
                if self._num_batches is not None and num_items >= self._num_batches:
                    return
                try:
                    batch = next(data_iter)
                except StopIteration:
                    break  # We've exhausted the dataset. Create a new iterator and start over.
                num_items += 1
                yield jax.tree.map(lambda x: jax.make_array_from_process_local_data(self._sharding, x), batch)


def _collate_fn(items):
    """Collate the batch elements into batched numpy arrays."""
    # Make sure to convert to numpy arrays before stacking since some of the incoming elements
    # may be JAX arrays.
    return jax.tree.map(lambda *x: np.stack(np.asarray(x), axis=0), *items)


def _worker_init_fn(worker_id: int) -> None:
    """Tell JAX inside the worker process not to preallocate the GPU memory."""
    # NOTE: This is called after jax is imported inside the worker process. This
    # means that this approach will not work for selecting the backend.
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

sharding = None
shuffle = False
num_batches = None
num_workers = 0
data_loader = TorchDataLoader(
    jax_dataset_trans,
    local_batch_size=1,
    sharding=sharding,
    shuffle=shuffle,
    num_batches=num_batches,
    num_workers=num_workers,
    seed=config.seed,
)

class DataLoader(Protocol[T_co]):
    """Interface for a data loader."""

    def data_config(self) -> _config.DataConfig:
        """Get the data config for this data loader."""
        raise NotImplementedError("Subclasses of DataLoader should implement data_config.")

    def __iter__(self) -> Iterator[T_co]:
        raise NotImplementedError("Subclasses of DataLoader should implement __iter__.")


class DataLoaderImpl(DataLoader):
    def __init__(self, data_config: _config.DataConfig, data_loader: TorchDataLoader):
        self._data_config = data_config
        self._data_loader = data_loader

    def data_config(self) -> _config.DataConfig:
        return self._data_config

    def __iter__(self):
        for batch in self._data_loader:
            yield _model.Observation.from_dict(batch), batch["actions"]

jax_loader = DataLoaderImpl(data_config, data_loader)

In [67]:
inputs, labels = next(iter(jax_loader))

In [68]:
inputs.state.shape

(1, 32)

## torch

In [69]:
data_config.norm_stats

{'state': NormStats(mean=array([ 0.06137776, -0.11410107,  0.50186026,  1.21013367, -0.02636181,
         1.5830574 ,  0.05373131, -0.65109986,  0.9976843 ,  1.74398351,
        -2.00823259, -1.00430524,  1.74444032,  0.56320119, -0.0033571 ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ]), std=array([0.15815794, 0.10150439, 0.05503138, 2.82920384, 0.10528079,
        0.41227192, 0.02931974, 0.29692534, 0.16840713, 0.2437624 ,
        0.38815749, 0.195886  , 0.16290529, 0.39232841, 0.99999434,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        ]), q01=array([-3.04437412e-01, -4.01245822e-01,  3.39070335e

In [70]:
torchdataset.meta.stats['state']['mean'] = data_config.norm_stats['state'].mean[:15]
torchdataset.meta.stats['state']['std'] = data_config.norm_stats['state'].std[:15]
torchdataset.meta.stats['joint_actions']['std'] = data_config.norm_stats['actions'].std[:8]
torchdataset.meta.stats['joint_actions']['mean'] = data_config.norm_stats['actions'].mean[:8]

In [71]:
cfg.policy.pretrained_path = "/mnt/lustre-grete/usr/u12045/vla/pi0_torch_newcp"

In [72]:
logging.info("Creating policy")
policy = make_policy(
    cfg=cfg.policy,
    ds_meta=torchdataset.meta,
)

load pretrained policy
Loading weights from local directory


In [73]:
policy.config.empty_cameras = 1


## Check stats of both

In [74]:
policy.normalize_inputs.buffer_state.mean


Parameter containing:
tensor([ 0.0614, -0.1141,  0.5019,  1.2101, -0.0264,  1.5831,  0.0537, -0.6511,
         0.9977,  1.7440, -2.0082, -1.0043,  1.7444,  0.5632, -0.0034],
       device='cuda:0')

In [75]:
data_config.norm_stats['state'].mean


array([ 0.06137776, -0.11410107,  0.50186026,  1.21013367, -0.02636181,
        1.5830574 ,  0.05373131, -0.65109986,  0.9976843 ,  1.74398351,
       -2.00823259, -1.00430524,  1.74444032,  0.56320119, -0.0033571 ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ])

In [76]:
policy.normalize_targets.buffer_joint_actions.mean

Parameter containing:
tensor([-0.6347,  0.9994,  1.7313, -2.0021, -1.0166,  1.7541,  0.5674, -0.3388],
       device='cuda:0')

In [77]:
data_config.norm_stats['actions'].mean

array([-0.63466722,  0.99936348,  1.73125029, -2.00207305, -1.01656699,
        1.75413775,  0.56735384, -0.33882481,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ])

In [173]:
# Use jax resize code
torch_sample = torchdataset[100]
torch_sample['state'] = torch_sample['state'].to(device)
torch_sample['joint_actions'] = torch_sample['joint_actions'].to(device)
torch_sample['image'] = torch_sample['image'].unsqueeze(0).to(device)
torch_sample['wrist_image'] = torch_sample['wrist_image'].unsqueeze(0).to(device)
torch_sample['task'] = [torch_sample['task']]
torch_images, torch_img_masks, torch_state, torch_lang_tokens, torch_lang_masks, torch_actions = policy.just_to_get_data_after_norm(torch_sample)


# # Use torch resize code
# torch_sample = torchdataset[0]
# torch_sample['state'] = torch_sample['state'].to(device)
# torch_sample['joint_actions'] = torch_sample['joint_actions'].to(device)
# torch_sample['image'] = (torch_sample['image']*255).unsqueeze(0).to(torch.uint8)
# torch_sample['wrist_image'] = (torch_sample['wrist_image']*255).unsqueeze(0).to(torch.uint8)
# torch_sample['task'] = [torch_sample['task']]
# torch_images, torch_img_masks, torch_state, torch_lang_tokens, torch_lang_masks, torch_actions = policy.just_to_get_data_after_norm(torch_sample)

In [172]:
# jax_sample_norm = jax_dataset_trans[0]
select_idx = 100
for i, sample in enumerate(iter(jax_loader)):
    if i < select_idx:
        continue
    elif i > select_idx:
        break
    (inputs, jax_actions) = sample
    jax_images = inputs.images
    jax_img_masks = inputs.image_masks
    jax_state = inputs.state
    jax_lang_tokens = inputs.tokenized_prompt
    jax_lang_masks = inputs.tokenized_prompt_mask

In [174]:
import numpy as np
np.allclose(torch_state.cpu().numpy(), jax_state, atol=1e-3) # True 1e-4->False

True

In [175]:
np.allclose(torch_actions.cpu().numpy(), jax_actions, atol=1e-6) # True

True

In [176]:
# lang token
np.all(torch_lang_tokens.cpu().numpy() == jax_lang_tokens), np.all(torch_lang_masks.cpu().numpy() == jax_lang_masks)

(Array(True, dtype=bool), Array(True, dtype=bool))

In [177]:
# image
np.allclose(torch_images[0].permute(0,2,3,1).cpu().numpy(), jax_images["base_0_rgb"], atol=1e-7)

True

In [178]:
# image mask
np.all(torch_img_masks[0].cpu().numpy() == jax_img_masks["base_0_rgb"])

Array(True, dtype=bool)

In [179]:
# left wrist image
np.allclose(torch_images[1].permute(0, 2,3,1).cpu().numpy(), jax_images["left_wrist_0_rgb"], atol=1e-7)

True

In [180]:
# left wrist mask
np.all(torch_img_masks[1].cpu().numpy() == jax_img_masks["left_wrist_0_rgb"])

Array(True, dtype=bool)

In [181]:
# right wrist image
np.allclose(torch_images[2].permute(0,2,3,1).cpu().numpy(), jax_images["right_wrist_0_rgb"], atol=1e-20)

True

In [182]:
# right wrist mask
np.all(torch_img_masks[2].cpu().numpy() == jax_img_masks["right_wrist_0_rgb"])

Array(True, dtype=bool)